In [ ]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================
# --- Standard Library ---
import os
import sys


import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn.utils.deprecation")


from   pathlib import Path
if Path.cwd().name == 'notebook':
    os.chdir('..')
sys.path.append(os.getcwd())
             
# --- Third-Party ---

import joblib
from   joblib import delayed, Parallel
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt


# --- Local Imports  ---
from config.presentation_config import (
    set_nature_style, 
    SEEDS,
    RESULTS_ROOT,
    IMAGES_DIR,
    IMAGE_DF_PATH,
    MODEL_COLORS, 
    NAME_MAPPING_SHORT, 
    TECH_TO_PUB,
    NAME_MAPPING_LONG,
    technical_names_master,
    ANALYSIS_METRICS,
    REF_VALUES
)

IMAGES_DIR.mkdir(exist_ok=True)

# ==============================================================================
# 3. INITIALIZATION & CONFIGURATION
# ==============================================================================
set_nature_style()

# ==============================================================================
# 4. LOAD STABLE FEATURES
# ==============================================================================
stable_tech_features = None

for seed in SEEDS:
    path = RESULTS_ROOT / f'Results_{seed}/Datasets/Training_Test_Splits/features_to_keep.pkl'
    if path.exists():
        loaded_feats = joblib.load(path) 
        if stable_tech_features is None:
            stable_tech_features = loaded_feats 
        elif len(loaded_feats) != len(stable_tech_features):
            print(f"❌ Warning: Feature count mismatch in Seed {seed}")

if stable_tech_features is None:
    raise ValueError("Could not load features_to_keep.pkl from any seed.")

print(f"✅ Setup Complete. Mapping {len(stable_tech_features)} features using config.")

✅ Setup Complete. Mapping 49 features using config.


# Per-Model Hyperparameter Space Table Summary

In [ ]:
from src.notebook_func.hyperparameter_models_table import create_journal_hyperparameter_table

create_journal_hyperparameter_table(
    outputdir=IMAGES_DIR/Path('Table_of_Hyperparameters')
)

# Exploratory Dataset Description

## General Informations and Features Stability Across Seed Runs: 

In [ ]:
input_numpy_dir = Path("data/input/numpy files")

if input_numpy_dir.exists():
    name_files_names = [f for f in os.listdir(input_numpy_dir) if f.endswith('.npy')]
    one_file_path = input_numpy_dir / name_files_names[0]
    img_shape = np.load(one_file_path).shape
    
    alpha_count_lc = sum(1 for name in name_files_names if "alpha" in name.lower())
    beta_count_lc = sum(1 for name in name_files_names if "beta" in name.lower())
    total_raw = alpha_count_lc + beta_count_lc

    print("="*50)
    print(f"RAW DATASET SUMMARY")
    print(f"Total files: {len(name_files_names)}")
    print(f"Alpha: {alpha_count_lc} | Beta: {beta_count_lc}")
    print(f"Dimensions: {img_shape[0]}x{img_shape[1]} pixels")
    print("="*50)
else:
    print(f"⚠️ Warning: {input_numpy_dir} not found.")


if IMAGE_DF_PATH.exists():
    image_df = joblib.load(IMAGE_DF_PATH)
    
    alpha_count = len(image_df[image_df['label'] == 0])
    beta_count = len(image_df[image_df['label'] == 1])
    total_filtered = alpha_count + beta_count

    print(f"\nPOST-PROCESSING SUMMARY (PhD Azzarello Fabio Filters)")
    print(f"Total Cells: {total_filtered}")
    print(f"Alpha: {alpha_count} ({(100*alpha_count/total_filtered):.2f}%)")
    print(f"Beta:  {beta_count} ({(100*beta_count/total_filtered):.2f}%)")
    print(f"Alpha-to-Beta Ratio: {(100*alpha_count/beta_count):.2f}%")
else:
    print(f"⚠️ Warning: Processed dataframe not found at {IMAGE_DF_PATH}")


print("\n" + "="*60)
print("S2. FEATURE STABILITY CHECK")
print("="*60)

feature_subsets_raw = []
seeds_present = []

for seed in SEEDS:
    path = RESULTS_ROOT / f'Results_{seed}/Datasets/Training_Test_Splits/features_to_keep.pkl'
    
    if path.exists():
        feats_list = joblib.load(path)
        feature_subsets_raw.append(frozenset(feats_list))
        seeds_present.append(seed)

unique_subsets = set(feature_subsets_raw)

if len(unique_subsets) == 1:
    print(f"✅ SUCCESS: Feature selection is consistent across {len(seeds_present)} seeds.")
    
    stable_tech_features = list(next(iter(unique_subsets)))
    final_ordered_technical_features = [t for t in technical_names_master if t in stable_tech_features]
    final_ordered_publication_features = [TECH_TO_PUB[t] for t in final_ordered_technical_features]

    print(f"Total Active Features: {len(final_ordered_technical_features)}")
else:
    print(f"❌ FAILURE: Found {len(unique_subsets)} different feature subsets across seeds.")

## Exploratory Data Analysis 

In [ ]:
# ==============================================================================
# S3. EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATION
# ==============================================================================

# --- 1. Imports EDA functions ---
from src.notebook_func.eda_visualization_functions import (
    get_feature_dataframe,
    label_distribution,
    PCA_features_visualization,
    linear_separability_test_publication,
    unsupervised_clustering_analysis_one_row,
    fit_gmm,
    make_unified_summary_figure,
    make_figure2
)

# --- 2. Setup Output Directory ---
EDA_OUTPUT_DIR = IMAGES_DIR / 'EDA_Analysis_Figures'
EDA_OUTPUT_DIR.mkdir(exist_ok=True)
print(f"📁 Figures will be saved to: {EDA_OUTPUT_DIR}")

# ==============================================================================
# EXECUTION BLOCK
# ==============================================================================

if 'final_ordered_publication_features' in locals():
    
    # --- A. Data Preparation ---
    master_features_df = get_feature_dataframe(IMAGE_DF_PATH, final_ordered_publication_features)
    
    if master_features_df is not None:
        
        # --- B. Standard EDA Visualizations ---
        label_distribution(master_features_df)
        
        PCA_features_visualization(
            master_features_df, EDA_OUTPUT_DIR, 
            text_size=19, title_fontsize=22, pad=15,
            x_tick_fontsize=20, y_tick_fontsize=20,
            show=True, save=True
        )
        
        linear_separability_test_publication(
            master_features_df, EDA_OUTPUT_DIR, 
            title_fontsize=22, axis_label_fontsize=20,  
            tick_fontsize=18, legend_fontsize=18, table_fontsize=22,       
            show=True, save=True
        )
        
        unsupervised_clustering_analysis_one_row(
            master_features_df, EDA_OUTPUT_DIR, 
            super_title_fontsize=30, title_fontsize=26, label_fontsize=28,  
            tick_fontsize=24, legend_fontsize=25, 
            k_means=True, gmm=True, y_supertitle=1.1, pad=25,
            show=True, save=True
        )

        # --- C. Publication Figures (GMM Cluster Enrichment) ---
        print("\n" + "=" * 50)
        print("S3.8: GMM Publication Figures")
        print("=" * 50)
        
        # Fit the GMM model once to pass to the plotting functions
        cluster_labels, X_scaled, pca, pc1_scores = fit_gmm(master_features_df)
        
        make_unified_summary_figure(
            features_df=master_features_df,
            cluster_labels=cluster_labels,
            X_scaled=X_scaled,
            pca=pca,
            output_dir=EDA_OUTPUT_DIR,
            title_fontsize=22, label_fontsize=18, tick_fontsize=16,
            annot_fontsize=18, legend_fontsize=17, table_fontsize=18,
            show=True, save=True
        )
        
        make_figure2(
            features_df=master_features_df,
            cluster_labels=cluster_labels,
            pca=pca,
            output_dir=EDA_OUTPUT_DIR,
            show=True, save=True
        )
else:
    print("❌ Error: 'final_ordered_publication_features' not found. Please run S2 first.")

# Test and Stress Test Graphs Results

## Retrieving Model Performances and Plotting setup

In [ ]:
from src.notebook_func.results_retrieval import process_seed_optimized

from src.notebook_func.model_performance_evaluation import (
    combined_model_performance_analysis,
    generate_journal_table,
    plot_model_comparison,
    plot_model_comparison_table
)


#______________________________________________________________________________________
# Retrieving model performances

# 1. Run Parallel Processing across all seeds 
print(f"🚀 Processing {len(SEEDS)} seeds in parallel...")
all_records_nested = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_seed_optimized)(RESULTS_ROOT, seed, multiple_sources=False) 
    for seed in SEEDS
)

# 2. Create Master DataFrame 
final_records = [record for sublist in all_records_nested for record in sublist]
master_df = pd.DataFrame(final_records)

if not master_df.empty:
    # 3. Apply Global Mappings from Config 
    master_df['model_name'] = master_df['model_name'].replace(NAME_MAPPING_LONG)
    master_df['color'] = master_df['model_name'].map(MODEL_COLORS) # 

    print(f"✅ Successfully loaded {len(master_df)} records.")
    print(f"Unique Models Found: {master_df['model_name'].unique()}")
    
    if master_df['color'].isnull().any():
        missing = master_df[master_df['color'].isnull()]['model_name'].unique()
        print(f"⚠️ Warning: Missing colors for: {missing}")
else:
    print("❌ Error: No data loaded. Check RESULTS_ROOT directory structure.")


#______________________________________________________________________________________
# CONFIGURATION FOR SPECIFIC PLOTS

# 2. Stress Parameters
TEST_PATH               = IMAGES_DIR / 'Test Set Results'
STRESS_PATH             = IMAGES_DIR / 'Stress Test Results'
STRESS_and_TEST_PATH    = IMAGES_DIR / 'Stress&Test Sets Average Results'


MAX_PER_CLASS_DIFFERENCE = 0.2
MAX_STRESS_LEVEL         = 3

# 3. Model Ordering 
priority_models = [
    'All Models Voting',  
    'LightGBM',            
    'XGBoost', 
    'Stacking KNN', 
    'KNN',
    'Greedy Voting', 
    'Logistic Regression',
    'Random Forest',
    'Extra Trees'
]

if 'master_df' in locals() and not master_df.empty:
    # Get models actually present in the data
    existing_models = master_df['model_name'].unique().tolist()
    
    preferred_order = [
        'All Models Voting', 'Greedy Voting', 
        'Stacking GBM', 'Stacking KNN', 'Stacking Logistic',
        'LightGBM', 'XGBoost', 'Extra Trees', 'Random Forest', 
        'KNN', 'Logistic Regression'
    ]
    
    # Sort existing models based on preference
    MODELS_TO_PLOT = [m for m in preferred_order if m in existing_models]
    
    # Add any others that might have been missed
    remaining = [m for m in existing_models if m not in MODELS_TO_PLOT]
    MODELS_TO_PLOT.extend(remaining)
    
    print(f"✅ Configured {len(MODELS_TO_PLOT)} models for plotting.")
    print("   Plot Order:", MODELS_TO_PLOT)
else:
    print("⚠️ Warning: 'master_df' not found. Cannot determine models to plot.")

Test and Stress Test Graphs

In [ ]:
print("\n=== Generating Test Set Results (Level 0) ===")
robust_models_names = combined_model_performance_analysis(
    df = master_df, 
    metrics = ANALYSIS_METRICS,
    ref_values = REF_VALUES,
    stress_level=0, 
    selected_models=priority_models[:5], 
    max_per_class_diff=0.2, 
    
    # --- LAYOUT SETTINGS ---
    fig_height_multiplier=1.5, 
    hspace=0.7,                 
    wspace=0.2,  
    section_gap_height = 1.5,     
    middle_header_height = 0.7,
    top_margin = 0.88,
    boxes_linewidth = 2,
    references_linewiths = 4,
    legend_line_width = 5,
    ball_size = 10,

    # --- NEW SPACING CONTROLS ---
    legend_gap_from_graph = 1.0,  # Increase to move legends further down from graph
    legend_intra_gap = 0.05,      # Decrease to move Legend 2 closer to Legend 1

    title_fontsize = 24,  
    header_fontsize = 30,           
    tick_fontsize = 25, 
    legend_fontsize = 25,    
    title_pad=50,             
    save=True, show=False, 
    output_dir = TEST_PATH
)

for lvl in range(1, MAX_STRESS_LEVEL + 1):
    print(f"\n=== Generating Stress Test Results (Level {lvl}) ===")
    combined_model_performance_analysis(
        df = master_df, 
        metrics = ANALYSIS_METRICS,
        ref_values = REF_VALUES,
        stress_level=lvl, 
        selected_models=priority_models[:5], 
        max_per_class_diff=1.0, 
        
        # --- LAYOUT SETTINGS ---
        fig_height_multiplier=1.5, 
        hspace=0.7,                 
        wspace=0.2,  
        section_gap_height = 1.5,     
        middle_header_height = 0.7,
        top_margin = 0.88,
        boxes_linewidth = 2,
        references_linewiths = 4,
        legend_line_width = 5,

        # --- NEW SPACING CONTROLS ---
        legend_gap_from_graph = 1.0,  # Increase to move legends further down from graph
        legend_intra_gap = 0.05,      # Decrease to move Legend 2 closer to Legend 1

        title_fontsize = 24,  
        header_fontsize = 30,           
        tick_fontsize = 25, 
        legend_fontsize = 25,    
        title_pad=50,             
        save=True, show=False, 
        output_dir=STRESS_PATH
    )

print("\n=== Generating Average Results ===")
combined_model_performance_analysis(
    df = master_df, 
    metrics = ANALYSIS_METRICS,
    ref_values = REF_VALUES,
    stress_level=None, 
    selected_models=priority_models[:5], 
    max_per_class_diff=0.2, 
    
    # --- LAYOUT SETTINGS ---
    fig_height_multiplier=1.5, 
    hspace=0.7,                 
    wspace=0.2,  
    section_gap_height = 1.5,     
    middle_header_height = 0.7,
    top_margin = 0.88,
    boxes_linewidth = 2,
    references_linewiths = 4,
    legend_line_width = 5,
    ball_size = 5,

    # --- NEW SPACING CONTROLS ---
    legend_gap_from_graph = 1.0, 
    legend_intra_gap = 0.05,     

    title_fontsize = 24,  
    header_fontsize = 30,           
    tick_fontsize = 25, 
    legend_fontsize = 25,    
    title_pad=50,             
    save=True, show=False, 
    output_dir=STRESS_and_TEST_PATH
)

Test and Stress Test Tables

In [ ]:
print("\n=== Generating Tables (TEST) ===")
generate_journal_table(
    analysis_df=master_df,   
    metrics=ANALYSIS_METRICS,    
    consistent_order=priority_models[:9],
    ref_values=REF_VALUES,             
    stress_level=0,                 
    font_size=22,         
    header_fontsize=20,    
    save=True,
    show=False,
    output_dir=TEST_PATH)

for lvl in range(1, MAX_STRESS_LEVEL + 1):
    print(f"\n=== Generating Stress Test Results (Level {lvl}) ===")
    generate_journal_table(
        analysis_df=master_df,   
        metrics=ANALYSIS_METRICS,    
        consistent_order=priority_models[:5],
        ref_values=REF_VALUES,             
        stress_level=lvl,                 
        font_size=22,         
        header_fontsize=20,    
        save=True,
        show=False, 
        output_dir=STRESS_PATH)

print("\n=== Generating Tables (STRESS+TEST) ===")
generate_journal_table(
    analysis_df=master_df,   
    metrics=ANALYSIS_METRICS,    
    consistent_order=priority_models[:9],
    ref_values=REF_VALUES,             
    stress_level=None,                 
    font_size=20,         
    header_fontsize=20,    
    save=True,
    show=False,
    output_dir=STRESS_and_TEST_PATH)


LightGBM and All_models_voting Comparison

In [ ]:
# Define the two models you want to compare
models_pair = [priority_models[0], priority_models[1]]

print(f"\n=== Running Pairwise Comparison: {models_pair[0]} vs {models_pair[1]} ===")
plot_model_comparison(
    master_df,
    models_to_compare=models_pair,
    stress_level=0, 
    title_fontsize=20,
    axis_fontsize=24,
    stat_fontsize=26,
    save=True,
    show=True,
    wspace = 0.3,
    hspace = 0.4,
    output_dir=TEST_PATH
)


plot_model_comparison_table(
    master_df,
    models_to_compare=models_pair,
    stress_level=0,
    title_fontsize=18,
    header_fontsize=20,
    cell_fontsize=18,
    save=True,
    show=True,
    output_dir=TEST_PATH
)

## Drop In Performances Analysis in Stress Test

In [ ]:
# ==============================================================================
# 1. ROBUSTNESS GRAPHS (Bar Charts of Performance Drops)
# ==============================================================================

from src.notebook_func.model_performance_evaluation import (
    plot_robustness_drops_graphs,
    plot_robustness_drops_table,
    plot_separated_model_analysis_final,
    plot_forest_only_refined,
    master_performance_analysis
)

outputdir = STRESS_and_TEST_PATH / Path('Drop_In_Performances_plots')
outputdir.mkdir(parents=True,exist_ok=True)

# Run Graph
plot_robustness_drops_graphs(
    master_df, 
    ANALYSIS_METRICS, 
    MODEL_COLORS, 
    OUTPUT_DIR=outputdir, 
    name_mapping=NAME_MAPPING_SHORT, 
    title_fontsize=20,
    axis_fontsize=18,
    ball_fontsize=18,
    legend_fontsize=20,
    save=True, 
    show=True
)

# Run Table
plot_robustness_drops_table(
    master_df, 
    ANALYSIS_METRICS, 
    OUTPUT_DIR=outputdir,
    ROW_HEIGHT = 0.09,
    offset = 0.02,
    save=False, 
    show=True,
    cell_fontsize=22,

    header_fontsize=22,
    row_label_fontsize=22
)


plot_separated_model_analysis_final(
    master_df=master_df, 
    model_name="LightGBM", 
    metrics_to_plot=ANALYSIS_METRICS, 
    ref_values=REF_VALUES,
    save=True,             
    show=True,             
    point=False,            
    eq_margin_multiplier=0.5,
    offset=-0.04,
    title_fontsize=24,
    axis_label_fontsize=22,
    tick_fontsize=18,
    legend_fontsize=20,
    table_header_fontsize=20,
    table_cell_fontsize=20,
    OUTPUT_DIR=outputdir
)


plot_forest_only_refined(
    master_df=master_df, 
    model_name="LightGBM",
    metrics_to_plot=ANALYSIS_METRICS,
    point=True,
    save=True,
    OUTPUT_DIR=outputdir
)

robust_models_names = master_performance_analysis(
    df=master_df, 
    metrics=ANALYSIS_METRICS,
    ref_values=REF_VALUES,
    stress_level=0, 
    selected_models=priority_models[:5], 
    target_robustness_model="LightGBM", 
    max_per_class_diff=0.2, 
    
    # --- LAYOUT SETTINGS (THE FIXES) ---
    fig_height_multiplier=1.25,   # Increased from 0.85 to give vertical breathing room
    section_gap_height=1.2,      # Pushes headers and legends away from plots
    hspace=0.8,                  # Pushes Row 2 (D,E,F) down so Row 1 X-labels fit
    wspace=0.3,  
    middle_header_height=0.7,
    top_margin=0.92,             # Lowers the top plots away from the main Suptitle
    legend_gap_from_graph=0.8,     

    # --- LINE WIDTHS E SIZES ---
    boxes_linewidth=2.0,        
    references_linewiths=4.0,
    legend_line_width=5.0,
    robustness_ref_linewidth=5.0,         
    ball_size=10.0,

    # --- FONT SIZES E PADDING ---
    header_fontsize=30,           
    label_fontsize=28,           
    title_fontsize=26,           
    tick_fontsize=20, 
    legend_fontsize=22,    
    title_pad=30,
    robustness_plot_width_ratio = 0.85, 
    
    # --- OUTPUT ---
    save=False, 
    show=True,                 
    output_dir=TEST_PATH
)

# Model Interpretability 

Imports

In [ ]:
# ==============================================================================
# S4. MODEL INTERPRETABILITY & EXPLAINABILITY
# ==============================================================================
# --- 1. Imports ---
from config.presentation_config import (
    NAME_MAPPING_SHORT, IMAGES_DIR, SEEDS, 
    TECH_TO_PUB, RESULTS_ROOT
)
from src.notebook_func.model_interpretability_funcs import (
    load_data_for_seed, get_gini_importance, analyze_permutation_importance_across_seeds,
    compute_pdp_safe_production, plot_combined_figure_decoupled, plot_combined_figure_side_by_side,
    plot_robust_3d_landscape, plot_gradient_histogram, combine_images_on_canvas_aligned,
    compute_and_save_shap_data, plot_shap_aligned_custom, save_individual_dependence_plots,
    plot_ultimate_interpretability_figure
)

# --- 2. Configuration ---
MODEL_TO_ANALYZE = 'LGBM'
DISPLAY_NAME = NAME_MAPPING_SHORT.get(MODEL_TO_ANALYZE, MODEL_TO_ANALYZE)
MODEL_INTERP_PATH = IMAGES_DIR / "Model Interpretability"
MODEL_INTERP_PATH.mkdir(parents=True, exist_ok=True)
print(f"📁 Figures will be saved to: {MODEL_INTERP_PATH.resolve()}")

MY_INTERESTING_FEATURES = [
    'ltp_u_7', 'ltp_u_4', 'ltp_u_5', 
    'shp_max_defect_depth', 'hu_2', 'inertia_eigval1'
]

# Ensure stable_tech_features exists from Step S2
if 'stable_tech_features' not in locals():
    raise ValueError("❌ Error: 'stable_tech_features' is missing. Run Step S2 first.")

SEEDS = range(3)

PHASE 1: Gini, Permutation, and PDP Analysis

In [ ]:
# ==============================================================================
# PHASE 1: Gini, Permutation, and PDP Analysis
# ==============================================================================
print("\n--- Phase 1: Global Feature Importance ---")

gini_summary, gini_raw = get_gini_importance(
    model_name=DISPLAY_NAME, seeds=SEEDS, tech_names=stable_tech_features, 
    save_dir=MODEL_INTERP_PATH, tech_to_pub_map=TECH_TO_PUB,    
    show_=True, save_=True, figsize=(16, 8)
)

top_n_features = analyze_permutation_importance_across_seeds(
    tech_names=stable_tech_features, tech_to_pub_map=TECH_TO_PUB,          
    model_name=DISPLAY_NAME, folder_name=RESULTS_ROOT, seeds=SEEDS, 
    top_n=6, n_repeats=5, save_dir=MODEL_INTERP_PATH, save_plot=True, show_=True
)

pdp_summary = compute_pdp_safe_production(
    model_name=MODEL_TO_ANALYZE, seeds=SEEDS,
    tech_features_list=stable_tech_features, save_dir=MODEL_INTERP_PATH
)

if (MODEL_INTERP_PATH / f"{MODEL_TO_ANALYZE}_PDP_Summary.csv").exists():
    plot_combined_figure_decoupled(
        perm_summary_path=MODEL_INTERP_PATH / f"{DISPLAY_NAME}_Permutation_Summary_Top6.csv",
        perm_raw_path=MODEL_INTERP_PATH / f"{DISPLAY_NAME}_Permutation_Raw_Seeds_Top6.csv",
        pdp_summary_path=MODEL_INTERP_PATH / f"{MODEL_TO_ANALYZE}_PDP_Summary.csv",
        mapping_dict=TECH_TO_PUB, custom_pdp_features=MY_INTERESTING_FEATURES,
        save_path=MODEL_INTERP_PATH / f"{MODEL_TO_ANALYZE}_PDP_Decoupled.png",
        show_=True, save=True
    )


PHASE 2: Ensemble 3D Landscape & Confidence

In [ ]:
# ==============================================================================
# PHASE 2: Ensemble 3D Landscape & Confidence
# ==============================================================================
print("\n--- Phase 2: Ensemble Decision Landscape ---")

models_list, X_tests_list, y_true_list = [], [], []

for seed in SEEDS:
    X_test_df, y_test = load_data_for_seed(RESULTS_ROOT, seed, 'test')
    model_path = RESULTS_ROOT / f"Results_{seed}/Results/Final_Results/Test_Results/{MODEL_TO_ANALYZE}_final_model.pkl"
    
    if X_test_df is not None and model_path.is_file():
        models_list.append(joblib.load(model_path))
        X_tests_list.append(X_test_df)
        y_true_list.append(y_test)

if models_list and X_tests_list:
    global_X = pd.concat(X_tests_list, axis=0, ignore_index=True)
    global_y = np.concatenate(y_true_list, axis=0)

    file_3d = MODEL_INTERP_PATH / f"3D_Robust_Ensemble_Landscape_{MODEL_TO_ANALYZE}.png"
    plot_robust_3d_landscape(
        models=models_list, global_X_scaled=global_X, global_y=global_y,
        resolution=100, scatter=False, save_path=file_3d
    )

    file_hist = MODEL_INTERP_PATH / f"Confidence_Histogram_{MODEL_TO_ANALYZE}.png"
    plot_gradient_histogram(models=models_list, X_scaled_list=X_tests_list, save_path=file_hist)

    combine_images_on_canvas_aligned(
        image_path_1=file_3d, image_path_2=file_hist,
        save_path=MODEL_INTERP_PATH / f"Ensemble_Analysis_Composite_{MODEL_TO_ANALYZE}.png", 
        title=f"Ensemble Analysis: {MODEL_TO_ANALYZE}", spacer_width=0.1 
    )


PHASE 3: SHAP Values & Ultimate Master Figure

In [ ]:
# ==============================================================================
# PHASE 3: SHAP Values & Ultimate Master Figure
# ==============================================================================
print("\n--- Phase 3: SHAP Values ---")

shap_vals, X_shap = compute_and_save_shap_data(
    model_name=MODEL_TO_ANALYZE, seeds=SEEDS, save_dir=MODEL_INTERP_PATH
)

if shap_vals is not None:
    plot_shap_aligned_custom(
        shap_values=shap_vals, X_df=X_shap, mapping_dict=TECH_TO_PUB,
        save_path=MODEL_INTERP_PATH / f"SHAP_Composite_Aligned_{MODEL_TO_ANALYZE}.png"
    )
    
    save_individual_dependence_plots(
        model_to_analyze=MODEL_TO_ANALYZE, shap_values=shap_vals, X_df=X_shap,
        mapping_dict=TECH_TO_PUB, save_dir=MODEL_INTERP_PATH, top_n=5
    )

    plot_ultimate_interpretability_figure(
        perm_summary_path=MODEL_INTERP_PATH / f"{DISPLAY_NAME}_Permutation_Summary_Top6.csv",
        perm_raw_path=MODEL_INTERP_PATH / f"{DISPLAY_NAME}_Permutation_Raw_Seeds_Top6.csv",
        pdp_summary_path=MODEL_INTERP_PATH / f"{MODEL_TO_ANALYZE}_PDP_Summary.csv",
        custom_pdp_features=MY_INTERESTING_FEATURES, top_n_perm=10,
        shap_values=shap_vals, X_shap_df=X_shap, mapping_dict=TECH_TO_PUB,
        save_path=MODEL_INTERP_PATH / f"ULTIMATE_Interpretability_{MODEL_TO_ANALYZE}.png"
    )

# On the importance of Augmentation Regime

In [ ]:
# ==============================================================================
# S5. AUGMENTATION REGIME JUSTIFICATION
# ==============================================================================
from src.notebook_func.augmentations_analysis import run_augmentation_regime_analysis

# Config
AUG_OUTPUT_DIR = IMAGES_DIR / "Augmentation Analysis"

if 'image_df' not in globals():
    image_df_path = RESULTS_ROOT / Path('image_df.pkl')
    image_df = joblib.load(image_df_path)

# Execute
run_augmentation_regime_analysis(
    image_df=image_df,
    output_dir=AUG_OUTPUT_DIR,
    n_samples=100, 
    tech_names=technical_names_master,
    save=True,
    show=True
)

Rerunning the Test and Stress Analysis on the New NO_AUG_ROOT or in other words new no augmentation configuration experiment setup

In [ ]:
# ==============================================================================
# S5. BASELINE PERFORMANCE & AUGMENTATION JUSTIFICATION (NO AUGMENTATION)

# --- 1. Imports ---
from config.presentation_config import (
    NAME_MAPPING_LONG, NAME_MAPPING_SHORT, MODEL_COLORS, 
    ANALYSIS_METRICS, REF_VALUES, IMAGES_DIR, SEEDS
)

from src.notebook_func.results_retrieval import process_seed_optimized
from src.notebook_func.model_performance_evaluation import (
    combined_model_performance_analysis,
    generate_journal_table,
    plot_robustness_drops_graphs, 
    plot_robustness_drops_table
)

# --- 2. Configuration for Baseline ---
NO_AUG_ROOT = Path("Dataset++_No_Aug")
NO_AUG_IMAGES_DIR = IMAGES_DIR / "Baseline_No_Aug_Results"
AUG_OUTPUT_DIR = NO_AUG_IMAGES_DIR / "Augmentation_Analysis"
NO_AUG_STRESS_PATH = NO_AUG_IMAGES_DIR / "Stress_Test_Results"

# Create directories
for path in [NO_AUG_IMAGES_DIR, AUG_OUTPUT_DIR, STRESS_PATH]:
    path.mkdir(parents=True, exist_ok=True)


Load Baseline Results (Dataset++_No_Aug)

In [ ]:

# ==============================================================================
# PHASE B: Load Baseline Results (Dataset++_No_Aug)
# ==============================================================================
print(f"\n--- Phase B: Loading Baseline Results from {NO_AUG_ROOT} ---")
print(f"🚀 Processing {len(SEEDS)} seeds in parallel...")

all_records_nested = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_seed_optimized)(NO_AUG_ROOT, seed, multiple_sources=False) 
    for seed in SEEDS
)

# Create Master DataFrame for No-Augmentation data
final_records = [record for sublist in all_records_nested for record in sublist]
no_aug_master_df = pd.DataFrame(final_records)

if not no_aug_master_df.empty:
    no_aug_master_df['model_name'] = no_aug_master_df['model_name'].replace(NAME_MAPPING_LONG)
    no_aug_master_df['color'] = no_aug_master_df['model_name'].map(MODEL_COLORS)

    print(f"✅ Successfully loaded {len(no_aug_master_df)} baseline records.")
    
    if no_aug_master_df['color'].isnull().any():
        missing = no_aug_master_df[no_aug_master_df['color'].isnull()]['model_name'].unique()
        print(f"⚠️ Warning: Missing colors for: {missing}")
else:
    raise ValueError(f"❌ Error: No data loaded. Ensure '{NO_AUG_ROOT}' exists and contains results.")


Baseline Performance Evaluation (Stress Level 3)

In [ ]:
# ==============================================================================
# PHASE C: Baseline Performance Evaluation (Stress Level 3)
# ==============================================================================
print("\n--- Phase C: Baseline Model Performance Evaluation ---")

lvl = 3
priority_models = [
    'All Models Voting', 'LightGBM', 'XGBoost', 'Stacking KNN', 'KNN',
    'Greedy Voting', 'Logistic Regression', 'Random Forest', 'Extra Trees'
]

# Ensure we only plot models that successfully loaded
existing_models = no_aug_master_df['model_name'].unique().tolist()
MODELS_TO_PLOT = [m for m in priority_models if m in existing_models]
MODELS_TO_PLOT.extend([m for m in existing_models if m not in MODELS_TO_PLOT])

print(f"\n=== Generating Baseline Stress Test Results (Level {lvl}) ===")

# 1. Generate Performance Plot
combined_model_performance_analysis(
    df=no_aug_master_df, 
    metrics=ANALYSIS_METRICS,
    ref_values=REF_VALUES,
    stress_level=lvl, 
    selected_models=MODELS_TO_PLOT[:5], 
    max_per_class_diff=1.0, 
    
    # --- LAYOUT SETTINGS ---
    fig_height_multiplier=1.5, hspace=0.7, wspace=0.2,  
    section_gap_height=1.5, middle_header_height=0.7, top_margin=0.88,
    boxes_linewidth=2, references_linewiths=4, legend_line_width=5,
    legend_gap_from_graph=1.0, legend_intra_gap=0.05, 

    title_fontsize=24, header_fontsize=30, tick_fontsize=25, 
    legend_fontsize=25, title_pad=50,             
    
    save=True, show=False, 
    output_dir=NO_AUG_STRESS_PATH
)

# 2. Generate Journal Table
generate_journal_table(
    analysis_df=no_aug_master_df,   
    metrics=ANALYSIS_METRICS,    
    consistent_order=MODELS_TO_PLOT[:5],
    ref_values=REF_VALUES,             
    stress_level=lvl,                 
    font_size=22, header_fontsize=20,    
    save=True, show=False, 
    output_dir=NO_AUG_STRESS_PATH
)


Robustness Drops Analysis

In [ ]:

# ==============================================================================
# PHASE D: Robustness Drops Analysis
# ==============================================================================
print("\n--- Phase D: Robustness Degradation Analysis ---")

# Run Drop Graph
plot_robustness_drops_graphs(
    master_df=no_aug_master_df, 
    ANALYSIS_METRICS=ANALYSIS_METRICS, 
    COLOR_CODEX=MODEL_COLORS, 
    OUTPUT_DIR=NO_AUG_IMAGES_DIR, 
    name_mapping=NAME_MAPPING_SHORT, 
    title_fontsize=20, axis_fontsize=18, ball_fontsize=18, legend_fontsize=20,
    save=True, show=True
)

# Run Drop Table
plot_robustness_drops_table(
    master_df=no_aug_master_df, 
    ANALYSIS_METRICS=ANALYSIS_METRICS, 
    OUTPUT_DIR=NO_AUG_IMAGES_DIR, 
    ROW_HEIGHT=0.09, offset=0.02,
    cell_fontsize=22, header_fontsize=22, row_label_fontsize=22,
    save=True, show=True
)


# Are The Alpha cells that I inserted just noise or have some latent informative value? 

Performance Retrieval and Comparative Analysis

This section validates the integration of additional Alpha cell images (the Dataset++ evolution). It is critical to demonstrate that these previously excluded samples contribute meaningful physiological information rather than noise, and that their removal would lead to a loss of discriminative power in the model.

To achieve this, the analysis evaluates performance across three primary data configurations:
- Dataset++: Custom balancing through targeted Alpha cell augmentation.
- Dataset_SMOTE: Class balancing achieved through the Synthetic Minority Over-sampling Technique (SMOTE)
- Dataset: The original baseline configuration with a lower Alpha cell count.

To verify the effective "generalization power" of the resulting models and ensure they have not merely memorized specific data splits, we execute a direct performance cross-check in two strict settings:
 - Unique Image Cross-Validation: Testing the Dataset++ models on unique images from the original Dataset test sets that were completely absent from the Dataset++ training phase. This approach strictly avoids data leakage and ensures that performance metrics are not artificially inflated.
 - Unseen Alpha Classification: Testing original Dataset models on the specific Alpha cell subsets that only Dataset++ was trained to identify. This assesses how well models trained on limited Alpha data can generalize to physiological variations they were never exposed to during training or initial testing.
 

In [ ]:
# ==============================================================================
# S6. VALIDATING THE ADDITION OF ALPHA CELLS (CROSS-DATASET GENERALIZABILITY)
# ==============================================================================

# --- 1. Imports ---
from config.presentation_config import NAME_MAPPING_LONG, IMAGES_DIR
from src.notebook_func.results_retrieval import process_seed_optimized
from src.notebook_func.cross_dataset_analysis import (
    generate_comparison_figure,
    run_comparison_and_get_data,
    create_quadrant_figure
)

# --- 2. Configuration ---
CROSS_DATA_OUT_DIR = IMAGES_DIR / "Cross_Dataset_Analysis"
CROSS_DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_PATHS = [Path('Dataset++'), Path('Dataset_SMOTE'), Path('Dataset')]
SEEDS = range(0, 10)

# Relative paths for loading data inside the comparison function
DATASET_A_FILE_REL_PATH = "Datasets/Training_Test_Splits/Global_Data/train/global_training_set.pkl"
DATASET_B_FILE_REL_PATH = "Datasets/Training_Test_Splits/Global_Data/test/test_set.pkl"

# ==============================================================================
# PHASE A: Load Master Data Across All Dataset Variations
# ==============================================================================
print("Loading data across all dataset variations...")
all_records = []

for source in SOURCE_PATHS:
    if not source.exists():
        print(f"⚠️ Skipping missing source: {source}")
        continue
        
    print(f"Processing source: {source}")
    results = Parallel(n_jobs=-1, verbose=1)(
        delayed(process_seed_optimized)(str(source), seed, multiple_sources=True) for seed in SEEDS
    )
    for r in results:
        all_records.extend(r)

master_cross_df = pd.DataFrame(all_records)
master_cross_df['model_name'] = master_cross_df['model_name'].replace(NAME_MAPPING_LONG)

# Filter to target model (LGBM)
lgbm_cross_df = master_cross_df[master_cross_df['model_name'] == 'LightGBM'].copy()
print(f"✅ Loaded {len(lgbm_cross_df)} LightGBM rows for cross-dataset analysis.")


In [ ]:
# ==============================================================================
# PHASE B: Intra-Dataset Performance (Test vs Stress Lvl 3)
# ==============================================================================
print("\n--- Generating Intra-Dataset Comparison Figure ---")

generate_comparison_figure(
    df=lgbm_cross_df, 
    boxplot_width=0.4, 
    tick_fontsize=17, 
    table_font_size=24,
    header_fontsize=18, 
    ylabel_fontsize=18,
    show=True
)


In [ ]:
# ==============================================================================
# PHASE C: Cross-Dataset Generalizability (Testing on Unique Images)
# ==============================================================================
print("\n--- Running Strict Cross-Dataset Generalizability Analysis ---")

# Define the comparisons: (Train Source, Test Source)
results_order = [
    (SOURCE_PATHS[0], SOURCE_PATHS[2]), # Dataset++ tested on Dataset
    (SOURCE_PATHS[2], SOURCE_PATHS[0]), # Dataset tested on Dataset++
    (SOURCE_PATHS[0], SOURCE_PATHS[1]), # Dataset++ tested on Dataset_SMOTE
    (SOURCE_PATHS[1], SOURCE_PATHS[0])  # Dataset_SMOTE tested on Dataset++
]

collected_results = []

for path_a, path_b in results_order:
    name_a, name_b, df_summary = run_comparison_and_get_data(
        dataset_a_path=path_a, 
        dataset_b_path=path_b,
        dataset_a_file_rel_path=DATASET_A_FILE_REL_PATH,
        dataset_b_file_rel_path=DATASET_B_FILE_REL_PATH,
        p_value_threshold=0.01,
        verbosity=1
    )
    collected_results.append((name_a, name_b, df_summary))

print("\n--- Generating Final Cross-Dataset Quadrant Figure ---")
create_quadrant_figure(
    data_collection=collected_results, 
    filename=str(CROSS_DATA_OUT_DIR / "Final_Article_Comparison_Table.png")
)

# On the Feasibility of RealTime Classification

In [ ]:
# ==============================================================================
# S7. ARCHITECTURAL SCALABILITY & REAL-TIME FEASIBILITY
# ==============================================================================


# --- 1. Consolidated Imports ---
from src.notebook_func.realtime_performance import (
    analyze_model_generalization, generate_comparative_scalability_dashboard, 
    generate_consolidated_dashboard, load_all_resources, run_robustness_check, 
    calculate_diversity_matrix, select_golden_trio, setup_engine_workspace, 
    run_scientific_benchmark, format_summary_table
)
from config.speed_benchmark_config import (
                        # BATCH SCALABILITY ANALYSIS
                        N_AUG, N_WARMUP, N_REP,
                        # OldvsNEW RealTIme Engine 
                        BENCH_N_REP, BENCH_N_FRAMES, BENCH_N_STAB_FRAMES
)

# --- 2. Global Configuration ---
RESULTS_ROOT = Path("Dataset++")
MODEL_TO_ANALYZE = 'LGBM'
PERFORMANCE_OUT_DIR = IMAGES_DIR / "Speed_Testing_Real_Time_Pipeline"
PERFORMANCE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# PART 1: BATCH SCALABILITY ANALYSIS (Serial vs Parallel)
# ==============================================================================
print("\n" + "="*60 + "\nPART 1: BATCH SCALABILITY ANALYSIS\n" + "="*60)

# Load basic requirements for scalability tests
image_df = joblib.load(RESULTS_ROOT / 'image_df.pkl')
feats_to_keep = joblib.load(RESULTS_ROOT / f"Results_{SEEDS[0]}/Datasets/Training_Test_Splits/features_to_keep.pkl")

all_models = []
for seed in SEEDS:
    path_to_model = RESULTS_ROOT / f"Results_{seed}/Results/Final_Results/Test_Results/{MODEL_TO_ANALYZE}_final_model.pkl"
    if path_to_model.is_file():
        all_models.append(joblib.load(path_to_model))

if all_models:
    # Run tests for both modes
    for m in ['serial', 'parallel']:
        analyze_model_generalization(
            image_df=image_df, all_models=all_models, selected_indices=feats_to_keep,
            save_dir=PERFORMANCE_OUT_DIR, n_augmentations=N_AUG, n_warmup=N_WARMUP, n_repeats=N_REP, mode=m
        )
        
    # Generate Dashboards
    generate_comparative_scalability_dashboard(
        save_dir=PERFORMANCE_OUT_DIR, fit_=False, ticks_fontsize=18, 
        axis_label_fontsize=20, title_label_fontsize=24, table_text_size=20
    )
    generate_consolidated_dashboard(save_dir=PERFORMANCE_OUT_DIR)
else:
    print("❌ No valid models found for scalability analysis.")


# ==============================================================================
# PART 2: REAL-TIME FEASIBILITY (Golden Trio & Optimized Engine)
# ==============================================================================
print("\n" + "="*60 + "\nPART 2: REAL-TIME ENGINE BENCHMARKING\n" + "="*60)

# --- Phase A: Ensemble Selection ---
print("\n--- Phase A: Selecting the Golden Trio ---")
models, scalers, selectors, test_sets, X_mega = load_all_resources(SEEDS, RESULTS_ROOT)

if len(models) >= 3:
    robustness_scores = run_robustness_check(models, test_sets)
    corr_matrix = calculate_diversity_matrix(models, X_mega, plot=False)
    GOLDEN_TRIO = select_golden_trio(robustness_scores, corr_matrix, models, printf_=True)
else:
    raise ValueError("❌ Not enough models loaded to form a trio. Check RESULTS_ROOT.")

# --- Phase B: Engine Workspace Deployment ---
print("\n--- Phase B: Deploying to Engine Workspace ---")
setup_engine_workspace(models, scalers, selectors, GOLDEN_TRIO)

# --- Phase C: Scientific Benchmark Execution ---
print("\n--- Phase C: Running Real-Time Feasibility Benchmark ---")
df_raw, df_stats = run_scientific_benchmark(
    n_repetitions=BENCH_N_REP, 
    n_frames=BENCH_N_FRAMES, 
    n_stability_frames=BENCH_N_STAB_FRAMES,   
    output_dir=PERFORMANCE_OUT_DIR, force_rerun=True, save_results=True,        
    title_fs=24, label_fs=22, tick_fs=18
)

# --- Phase D: Verification of Article Claims ---
if df_stats is not None:
    print("\n--- PERFORMANCE SUMMARY ---")
    print(format_summary_table(df_stats).to_string(index=False))
    
    # Extract values for the New Engine
    opt_med = df_stats.loc[df_stats['Engine'] == 'New Engine (RealTime)', 'Median_Mean'].values[0]
    mae = df_stats.loc[df_stats['Engine'] == 'New Engine (RealTime)', 'MAE_Mean'].values[0]
    agree = df_stats.loc[df_stats['Engine'] == 'New Engine (RealTime)', 'Agreement_Mean'].values[0]